In [ ]:
import cvxpy as cp
import pandas as pd
import networkx as nx

from typing import Dict, List, Tuple
import zipfile
import os

from tqdm import tqdm

In [ ]:
class CVXPYConfigurator:
    """Конфигуратор параметров CVXPY для разных сценариев"""
    
    @staticmethod
    def get_quality_profile():
        """
        Качественное решение, может быть медленнее
        Use case: Средние задачи, важно качество
        """
        return {
            'solver': cp.SCIP,
            'verbose': True,
            'scip_params': {
                'limits/gap': 0.01,
                'limits/time': 3600,
            }
        }
    
    @staticmethod
    def get_optimal_profile():
        """
        Поиск оптимума
        Use case: Малые задачи, нужен оптимум
        """
        return {
            'solver': cp.SCIP,
            'verbose': True,
            'scip_params': {
                'limits/gap': 0.0,
                'limits/time': 3600,
            }
        }
    
    @staticmethod
    def get_highs_profile():
        """
        Профиль для HiGHS solver (прямой интерфейс)
        Требует установки: pip install highspy
        """
        return {
            'solver': cp.CLARABEL,  # Временная заглушка, будет заменено на HIGHS
            'verbose': True,
            'time_limit_sec': 3600,
        }
    
    @staticmethod
    def get_cbc_profile():
        """
        Профиль для CBC solver (open-source)
        """
        return {
            'solver': cp.CBC,
            'verbose': True,
            'maximumSeconds': 3600,
            'ratioGap': 0.01,
        }
    
    @staticmethod
    def get_glpk_profile():
        """
        Профиль для GLPK_MI solver (open-source)
        """
        return {
            'solver': cp.GLPK_MI,
            'verbose': True,
            'glpk': {
                'tm_lim': 3600000,  # в миллисекундах
                'mip_gap': 0.01,
            }
        }



In [ ]:
class MultiCommodityFlowSolver:
    def __init__(self, 
        distance_matrix_file: str, 
        offices_file: str, 
        reqs_file: str, 
        max_paths_length_2: int = 50,
        solver_profile: str = 'cbc'  # 'quality', 'optimal', 'highs', 'cbc', 'glpk'
    ):
        """
        Инициализация решателя задачи multi-commodity flow
        
        Args:
            distance_matrix_file: путь к файлу с матрицей расстояний
            offices_file: путь к файлу с данными об офисах
            reqs_file: путь к файлу с запросами на перевозку
            max_paths_length_2: максимальное количество путей длины 2 для каждого товара
            solver_profile: 'quality', 'optimal', 'highs', 'cbc', 'glpk'
        """
    
        self.distance_df = pd.read_csv(distance_matrix_file)
        self.offices_df = pd.read_csv(offices_file)
        self.reqs_df = pd.read_csv(reqs_file)
        
        self.max_paths_length_2 = max_paths_length_2
        self.solver_profile = solver_profile
        
        self.vehicle_capacity = 90 
        self.transfer_cost_per_unit = 100 
        
        self.graph = self._build_graph()
        
        self.office_data = self._prepare_office_data()
        self.commodities = self._prepare_commodities()
        
        self.paths = self._generate_paths_length_2()
        
        self.problem = None
        self.x_vars = {}
        self.y_vars = {}
        self.status = None
        self.solve_time = None
        
        
    def _build_graph(self) -> nx.DiGraph:
        """Построение ориентированного графа из матрицы расстояний"""
        G = nx.DiGraph()
        
        for _, row in self.distance_df.iterrows():
            src = int(row['src'])
            dst = int(row['dst'])
            price = float(row['price'])
            
            G.add_edge(src, dst, weight=price, price=price)
        
        print(f"Граф создан: {G.number_of_nodes()} вершин, {G.number_of_edges()} рёбер")
        return G
    
    def _prepare_office_data(self) -> Dict:
        """Подготовка данных об офисах"""
        office_data = {}
        
        for _, row in self.offices_df.iterrows():
            office_id = int(row['office_id'])
            office_data[office_id] = {
                'name': row['office_name'],
                'transfer_max': float(row['transfer_max'])  # Максимальная складская ёмкость
            }
        
        print(f"Загружено {len(office_data)} офисов")
        return office_data
    
    def _prepare_commodities(self) -> List[Dict]:
        """Подготовка данных о товарах (commodities)"""
        commodities = []
        
        for idx, row in self.reqs_df.iterrows():
            commodity = {
                'id': idx,
                'src': int(row['src_office_id']),
                'dst': int(row['dst_office_id']),
                'demand': float(row['volume'])
            }
            commodities.append(commodity)
        
        print(f"Загружено {len(commodities)} товаров")
        return commodities
    
    def _generate_paths_length_2(self) -> Dict[int, List[Tuple[List[int], float]]]:
        """
        Генерация путей длины 2 для каждого товара
        Для каждого товара генерируем:
        1. Прямое ребро (если существует) - путь длины 1
        2. До max_paths_length_2 путей длины 2 (src -> intermediate -> dst)
        
        Пути длины 2 сортируются по стоимости и берутся лучшие max_paths_length_2
        
        Returns:
            Словарь {commodity_id: [(path, cost), ...]}
        """
        paths = {}
        
        for commodity in self.commodities:
            k = commodity['id']
            src = commodity['src']
            dst = commodity['dst']
            
            commodity_paths = []
            
            if self.graph.has_edge(src, dst):
                direct_path = [src, dst]
                direct_cost = self.graph[src][dst]['price']
                commodity_paths.append((direct_path, direct_cost))
                print(f"Товар {k} ({src}->{dst}): добавлен прямой путь, стоимость={direct_cost:.2f}")
            
            paths_length_2 = []
            
            for intermediate in self.graph.nodes():
                if intermediate == src or intermediate == dst:
                    continue
                
                if self.graph.has_edge(src, intermediate) and self.graph.has_edge(intermediate, dst):
                    path = [src, intermediate, dst]
                    cost = (self.graph[src][intermediate]['price'] + 
                           self.graph[intermediate][dst]['price'])
                    paths_length_2.append((path, cost))
            
            paths_length_2.sort(key=lambda x: x[1])
            selected_paths_length_2 = paths_length_2[:self.max_paths_length_2]
            
            commodity_paths.extend(selected_paths_length_2)
            
            if not commodity_paths:
                print(f"⚠️  ВНИМАНИЕ: Не найдено путей для товара {k} ({src} -> {dst})")
            else:
                num_direct = 1 if self.graph.has_edge(src, dst) else 0
                num_length_2 = len(selected_paths_length_2)
                print(f"Товар {k} ({src}->{dst}, объём={commodity['demand']}): "
                      f"{len(commodity_paths)} путей (прямых: {num_direct}, длины 2: {num_length_2})")
                
                if num_length_2 > 0:
                    min_cost = selected_paths_length_2[0][1]
                    max_cost = selected_paths_length_2[-1][1]
                    print(f"  Стоимость путей длины 2: мин={min_cost:.2f}, макс={max_cost:.2f}")
            
            paths[k] = commodity_paths
        
        return paths
    
    def _path_to_edges(self, path: List[int]) -> List[Tuple[int, int]]:
        """Преобразование пути в список рёбер"""
        return [(path[i], path[i+1]) for i in range(len(path)-1)]
    
    def _get_transit_nodes(self, path: List[int], src: int, dst: int) -> List[int]:
        """
        Получить транзитные узлы пути (исключая источник и сток)
        
        Args:
            path: путь как список вершин
            src: источник товара
            dst: сток товара
            
        Returns:
            Список транзитных узлов
        """
        transit = []
        for node in path:
            if node != src and node != dst:
                transit.append(node)
        return transit
    
    def build_model(self):
        """Построение модели оптимизации через CVXPY"""
        print("\n=== Построение модели CVXPY ===")
        
        print("\nСтатистика путей:")
        total_paths = 0
        total_direct = 0
        total_length_2 = 0
        
        for k, commodity_paths in self.paths.items():
            num_paths = len(commodity_paths)
            total_paths += num_paths
            
            num_direct = sum(1 for path, _ in commodity_paths if len(path) == 2)
            num_length_2 = sum(1 for path, _ in commodity_paths if len(path) == 3)
            
            total_direct += num_direct
            total_length_2 += num_length_2
            
            commodity = self.commodities[k]
            print(f"  Товар {k} ({commodity['src']}->{commodity['dst']}): "
                  f"{num_paths} путей (прямых: {num_direct}, длины 2: {num_length_2})")
        
        print(f"\n  ВСЕГО путей передается в солвер: {total_paths}")
        print(f"    - Прямых путей: {total_direct}")
        print(f"    - Путей длины 2: {total_length_2}\n")
        
        print("Создание переменных потока...")
        for k, commodity_paths in self.paths.items():
            for p_idx, (path, _) in enumerate(commodity_paths):
                var_name = f'x_k{k}_p{p_idx}'
                self.x_vars[(k, p_idx)] = cp.Variable(nonneg=True, name=var_name)
        
        print(f"  ✓ Создано {len(self.x_vars)} переменных потока")
        
        print("Создание переменных для ТС...")
        edges = set()
        for commodity_paths in self.paths.values():
            for path, _ in commodity_paths:
                edges.update(self._path_to_edges(path))
        
        for edge in edges:
            var_name = f'y_e_{edge[0]}_{edge[1]}'
            self.y_vars[edge] = cp.Variable(integer=True, nonneg=True, name=var_name)
        
        print(f"  ✓ Создано {len(self.y_vars)} переменных для ТС на {len(edges)} рёбрах")
        
        print("Создание целевой функции...")
        objective_terms = []
        
        for edge, y_var in self.y_vars.items():
            edge_cost = self.graph[edge[0]][edge[1]]['price']
            objective_terms.append(edge_cost * y_var)
        
        for k, commodity_paths in self.paths.items():
            src = self.commodities[k]['src']
            dst = self.commodities[k]['dst']
            
            for p_idx, (path, _) in enumerate(commodity_paths):
                transit_nodes = self._get_transit_nodes(path, src, dst)
                
                transfer_cost_coefficient = len(transit_nodes) * self.transfer_cost_per_unit
                
                if transfer_cost_coefficient > 0:
                    x_var = self.x_vars[(k, p_idx)]
                    objective_terms.append(transfer_cost_coefficient * x_var)
        
        objective = cp.Minimize(cp.sum(objective_terms))
        print("  ✓ Целевая функция создана")
        
        print("Создание ограничений...")
        constraints = []
        
        print("  1. Ограничения полного потока для каждого товара...")
        for k in tqdm(self.paths.keys()):
            demand = self.commodities[k]['demand']
            commodity_paths = self.paths[k]
            
            flow_sum = cp.sum([self.x_vars[(k, p_idx)] for p_idx in range(len(commodity_paths))])
            constraints.append(flow_sum == demand)
        
        print(f"     ✓ Создано {len(self.paths)} ограничений")
        
        print("  2. Связь потока и количества ТС...")
        for edge in tqdm(self.y_vars.keys()):
            edge_flow_terms = []
            
            for k, commodity_paths in self.paths.items():
                for p_idx, (path, _) in enumerate(commodity_paths):
                    if edge in self._path_to_edges(path):
                        edge_flow_terms.append(self.x_vars[(k, p_idx)])
            
            if edge_flow_terms:
                total_edge_flow = cp.sum(edge_flow_terms)
                constraints.append(total_edge_flow <= self.vehicle_capacity * self.y_vars[edge])
        
        print(f"     ✓ Создано {len(self.y_vars)} ограничений")
        
        print("  3. Ограничения складской ёмкости узлов...")
        node_constraint_count = 0
        
        for node_id, node_data in tqdm(self.office_data.items()):
            transfer_max = node_data['transfer_max']
            
            node_flow_terms = []
            
            for k, commodity_paths in self.paths.items():
                src = self.commodities[k]['src']
                dst = self.commodities[k]['dst']
                
                if node_id == src or node_id == dst:
                    continue
                
                for p_idx, (path, _) in enumerate(commodity_paths):
                    if node_id in path:
                        node_flow_terms.append(self.x_vars[(k, p_idx)])
            
            if node_flow_terms:
                total_node_flow = cp.sum(node_flow_terms)
                constraints.append(total_node_flow <= transfer_max)
                node_constraint_count += 1
        
        print(f"     ✓ Создано {node_constraint_count} ограничений")
        
        self.problem = cp.Problem(objective, constraints)
        
        print(f"\n✓ Модель CVXPY построена:")
        print(f"  Переменных: {len(self.x_vars) + len(self.y_vars)}")
        print(f"    - Потоков (непрерывные): {len(self.x_vars)}")
        print(f"    - ТС (целочисленные): {len(self.y_vars)}")
        print(f"  Ограничений: {len(constraints)}")
        
        return True
    
    def solve(self, time_limit_seconds: int = 60):
        """
        Решение задачи оптимизации
        
        Args:
            time_limit_seconds: ограничение по времени в секундах
        """
        if self.problem is None:
            print("ОШИБКА: Модель не построена")
            return False
        
        print(f"\n=== Решение задачи (лимит времени: {time_limit_seconds}s) ===")
        print(f"Профиль решателя: {self.solver_profile}")
        
        if self.solver_profile == 'optimal':
            config = CVXPYConfigurator.get_optimal_profile()
        elif self.solver_profile == 'highs':
            config = CVXPYConfigurator.get_highs_profile()
        elif self.solver_profile == 'cbc':
            config = CVXPYConfigurator.get_cbc_profile()
        elif self.solver_profile == 'glpk':
            config = CVXPYConfigurator.get_glpk_profile()
        else:
            config = CVXPYConfigurator.get_quality_profile()
        
        solver_name = config.pop('solver')
        verbose = config.pop('verbose', True)
        
        if self.solver_profile == 'highs':
            try:
                # Пробуем использовать прямой интерфейс HiGHS
                import highspy
                print("✓ Найден прямой интерфейс HiGHS (highspy)")
                
                # Создаем HiGHS модель напрямую
                return self._solve_with_highspy(time_limit_seconds, verbose)
                
            except ImportError:
                print("⚠️  Прямой интерфейс HiGHS (highspy) не установлен")
                print("   Попытка использовать HiGHS через CVXPY...")
                
                # Пробуем через CVXPY
                try:
                    # Проверяем доступность HiGHS в CVXPY
                    if hasattr(cp, 'CLARABEL'):
                        print("   Используем CLARABEL как альтернативу")
                        solver_name = cp.CLARABEL
                    else:
                        print("   Откатываемся на CBC")
                        solver_name = cp.CBC
                        config = CVXPYConfigurator.get_cbc_profile()
                        verbose = config.pop('verbose', True)
                        solver_name = config.pop('solver')
                except Exception as e:
                    print(f"   Ошибка: {e}")
                    print("   Используем CBC")
                    config = CVXPYConfigurator.get_cbc_profile()
                    verbose = config.pop('verbose', True)
                    solver_name = config.pop('solver')
        
        
    def _solve_with_highspy(self, time_limit_seconds: int, verbose: bool):
        """
        Решение напрямую через highspy (если установлен)
        """
        import highspy
        
        print("\n=== Использование прямого интерфейса HiGHS ===")
        
        h = highspy.Highs()
        
        # Настройка параметров HiGHS
        h.setOptionValue("time_limit", float(time_limit_seconds))
        h.setOptionValue("mip_rel_gap", 0.01)
        h.setOptionValue("log_to_console", verbose)
        h.setOptionValue("output_flag", verbose)
        
        print(f"Параметры HiGHS:")
        print(f"  time_limit: {time_limit_seconds}s")
        print(f"  mip_rel_gap: 0.01")
        
        # Создание переменных
        print("\nПодготовка модели для HiGHS...")
        
        # Индексы переменных
        var_indices = {}
        var_count = 0
        
        # Добавляем переменные потока (непрерывные)
        for (k, p_idx) in self.x_vars.keys():
            var_indices[(k, p_idx, 'x')] = var_count
            var_count += 1
        
        # Добавляем переменные ТС (целочисленные)
        for edge in self.y_vars.keys():
            var_indices[(edge, 'y')] = var_count
            var_count += 1
        
        print(f"  Переменных: {var_count}")
        
        # Нижние и верхние границы переменных
        col_lower = [0.0] * var_count
        col_upper = [highspy.kHighsInf] * var_count
        col_cost = [0.0] * var_count
        integrality = [highspy.HighsVarType.kContinuous] * var_count
        
        # Устанавливаем коэффициенты целевой функции
        # Стоимость ТС
        for edge, y_var in self.y_vars.items():
            edge_cost = self.graph[edge[0]][edge[1]]['price']
            idx = var_indices[(edge, 'y')]
            col_cost[idx] = edge_cost
            integrality[idx] = highspy.HighsVarType.kInteger
        
        # Стоимость перегруза
        for k, commodity_paths in self.paths.items():
            src = self.commodities[k]['src']
            dst = self.commodities[k]['dst']
            
            for p_idx, (path, _) in enumerate(commodity_paths):
                transit_nodes = self._get_transit_nodes(path, src, dst)
                transfer_cost = len(transit_nodes) * self.transfer_cost_per_unit
                
                if transfer_cost > 0:
                    idx = var_indices[(k, p_idx, 'x')]
                    col_cost[idx] = transfer_cost
        
        # Создаем модель
        h.addVars(var_count, col_lower, col_upper)
        h.changeColsCost(var_count, list(range(var_count)), col_cost)
        h.changeColsIntegrality(var_count, list(range(var_count)), integrality)
        
        # Добавляем ограничения
        print("  Добавление ограничений...")
        
        # 1. Ограничения потока (равенства)
        for k in self.paths.keys():
            demand = self.commodities[k]['demand']
            commodity_paths = self.paths[k]
            
            indices = [var_indices[(k, p_idx, 'x')] for p_idx in range(len(commodity_paths))]
            values = [1.0] * len(indices)
            
            h.addRow(demand, demand, len(indices), indices, values)
        
        # 2. Ограничения вместимости ТС
        for edge in self.y_vars.keys():
            indices = []
            values = []
            
            # Потоки через ребро
            for k, commodity_paths in self.paths.items():
                for p_idx, (path, _) in enumerate(commodity_paths):
                    if edge in self._path_to_edges(path):
                        indices.append(var_indices[(k, p_idx, 'x')])
                        values.append(1.0)
            
            # Вместимость ТС
            indices.append(var_indices[(edge, 'y')])
            values.append(-self.vehicle_capacity)
            
            if indices:
                h.addRow(-highspy.kHighsInf, 0.0, len(indices), indices, values)
        
        # 3. Ограничения складской ёмкости узлов
        for node_id, node_data in self.office_data.items():
            transfer_max = node_data['transfer_max']
            
            indices = []
            values = []
            
            for k, commodity_paths in self.paths.items():
                src = self.commodities[k]['src']
                dst = self.commodities[k]['dst']
                
                if node_id == src or node_id == dst:
                    continue
                
                for p_idx, (path, _) in enumerate(commodity_paths):
                    if node_id in path:
                        indices.append(var_indices[(k, p_idx, 'x')])
                        values.append(1.0)
            
            if indices:
                h.addRow(0.0, transfer_max, len(indices), indices, values)
        
        print("Модель готова")
        
        # Решение
        print("\nЗапуск HiGHS...")
        h.run()
        
        # Получение результатов
        solution = h.getSolution()
        model_status = h.getModelStatus()
        info = h.getInfo()
        
        print(f"\nСтатус HiGHS: {model_status}")
        
        if model_status == highspy.HighsModelStatus.kOptimal or model_status == highspy.HighsModelStatus.kTimeLimit:
            print("✓ Найдено оптимальное решение")
            print(f"  Целевая функция: {info.objective_function_value:.2f}")
            
            # Сохраняем решение в переменные CVXPY для совместимости
            for (k, p_idx), var in self.x_vars.items():
                idx = var_indices[(k, p_idx, 'x')]
                var._value = solution.col_value[idx]
            
            for edge, var in self.y_vars.items():
                idx = var_indices[(edge, 'y')]
                var._value = solution.col_value[idx]
            
            self.status = cp.OPTIMAL
            # Сохраняем значение целевой функции для совместимости
            self.problem._value = info.objective_function_value
            
            return True
        else:
            print(f"✗ Решение не найдено: {model_status}")
            self.status = cp.INFEASIBLE
            return False
    
    def get_results(self) -> Dict:
        """Получение результатов решения"""
        if self.status not in [cp.OPTIMAL, cp.OPTIMAL_INACCURATE]:
            return None
        
        results = {
            'objective_value': self.problem.value if hasattr(self.problem, 'value') else self.problem._value,
            'solve_time': self.solve_time,
            'flows': [],
            'vehicles': [],
            'node_usage': {},
            'summary': {}
        }
        
        total_flow_cost = 0
        total_transfer_cost = 0
        
        for (k, p_idx), var in self.x_vars.items():
            if hasattr(var, 'value'):
                flow_value = var.value
            else:
                flow_value = var._value if hasattr(var, '_value') else None
            
            if flow_value is not None and flow_value > 0.001:
                path, path_cost = self.paths[k][p_idx]
                commodity = self.commodities[k]
                
                transit_nodes = self._get_transit_nodes(path, commodity['src'], commodity['dst'])
                transfer_cost = len(transit_nodes) * self.transfer_cost_per_unit * flow_value
                
                for node in transit_nodes:
                    if node not in results['node_usage']:
                        results['node_usage'][node] = {
                            'total_flow': 0,
                            'max_capacity': self.office_data.get(node, {}).get('transfer_max', float('inf')),
                            'commodities': []
                        }
                    results['node_usage'][node]['total_flow'] += flow_value
                    results['node_usage'][node]['commodities'].append({
                        'commodity_id': k,
                        'flow': flow_value
                    })
                
                flow_info = {
                    'commodity_id': k,
                    'src': commodity['src'],
                    'dst': commodity['dst'],
                    'demand': commodity['demand'],
                    'path': path,
                    'path_length': len(path),
                    'flow': flow_value,
                    'edges_cost': sum(self.graph[path[i]][path[i+1]]['price'] for i in range(len(path)-1)),
                    'transit_nodes': transit_nodes,
                    'transfer_cost': transfer_cost
                }
                
                results['flows'].append(flow_info)
                total_transfer_cost += transfer_cost
        
        total_vehicle_cost = 0
        total_vehicles = 0
        
        for edge, var in self.y_vars.items():
            if hasattr(var, 'value'):
                vehicle_count = var.value
            else:
                vehicle_count = var._value if hasattr(var, '_value') else None
            
            if vehicle_count is not None and vehicle_count > 0.001:
                edge_cost = self.graph[edge[0]][edge[1]]['price']
                vehicle_cost = edge_cost * vehicle_count
                
                edge_flow = 0
                for (k, p_idx), flow_var in self.x_vars.items():
                    path, _ = self.paths[k][p_idx]
                    if edge in self._path_to_edges(path):
                        if hasattr(flow_var, 'value'):
                            flow_val = flow_var.value
                        else:
                            flow_val = flow_var._value if hasattr(flow_var, '_value') else None
                        if flow_val is not None:
                            edge_flow += flow_val
                
                vehicle_info = {
                    'edge': edge,
                    'count': vehicle_count,
                    'unit_cost': edge_cost,
                    'total_cost': vehicle_cost,
                    'total_flow': edge_flow,
                    'utilization': (edge_flow / (vehicle_count * self.vehicle_capacity) * 100) if vehicle_count > 0 else 0
                }
                
                results['vehicles'].append(vehicle_info)
                total_vehicle_cost += vehicle_cost
                total_vehicles += vehicle_count
        
        # Сводная информация
        results['summary'] = {
            'total_cost': results['objective_value'],
            'vehicle_cost': total_vehicle_cost,
            'transfer_cost': total_transfer_cost,
            'total_vehicles': total_vehicles,
            'num_active_flows': len(results['flows']),
            'num_active_edges': len(results['vehicles']),
            'num_transit_nodes': len(results['node_usage']),
            'solve_time': results['solve_time'] if results['solve_time'] else 0,
            'solver': 'CVXPY/HiGHS' if self.solver_profile == 'highs' else 'CVXPY',
            'solver_profile': self.solver_profile,
            'status': self.status
        }
        
        return results
    
    
    def export_results(self, results: Dict, output_prefix: str = 'results'):
        """Экспорт результатов в CSV файлы"""
        if results is None:
            print("Нет результатов для экспорта")
            return
        
        flows_data = []
        for flow in results['flows']:
            flows_data.append({
                'commodity_id': flow['commodity_id'],
                'src': flow['src'],
                'dst': flow['dst'],
                'demand': flow['demand'],
                'flow': flow['flow'],
                'path': ' -> '.join(map(str, flow['path'])),
                'path_length': flow['path_length'],
                'edges_cost': flow['edges_cost'],
                'transit_nodes': ', '.join(map(str, flow['transit_nodes'])),
                'num_transit_nodes': len(flow['transit_nodes']),
                'transfer_cost': flow['transfer_cost']
            })
        
        flows_df = pd.DataFrame(flows_data)
        flows_file = f'{output_prefix}_flows.csv'
        flows_df.to_csv(flows_file, index=False)
        print(f"✓ Потоки экспортированы в {flows_file}")
        
        # Экспорт ТС
        vehicles_data = []
        for vehicle in results['vehicles']:
            vehicles_data.append({
                'src': vehicle['edge'][0],
                'dst': vehicle['edge'][1],
                'vehicle_count': vehicle['count'],
                'unit_cost': vehicle['unit_cost'],
                'total_cost': vehicle['total_cost'],
                'total_flow': vehicle['total_flow'],
                'utilization_percent': vehicle['utilization']
            })
        
        vehicles_df = pd.DataFrame(vehicles_data)
        vehicles_file = f'{output_prefix}_vehicles.csv'
        vehicles_df.to_csv(vehicles_file, index=False)
        print(f"✓ ТС экспортированы в {vehicles_file}")
        
        nodes_data = []
        for node_id, usage in results['node_usage'].items():
            nodes_data.append({
                'node_id': node_id,
                'total_flow': usage['total_flow'],
                'max_capacity': usage['max_capacity'],
                'utilization_percent': (usage['total_flow'] / usage['max_capacity'] * 100) if usage['max_capacity'] > 0 else 0,
                'num_commodities': len(usage['commodities'])
            })
        
        nodes_df = pd.DataFrame(nodes_data)
        nodes_file = f'{output_prefix}_nodes.csv'
        nodes_df.to_csv(nodes_file, index=False)
        print(f"✓ Использование узлов экспортировано в {nodes_file}")
        
        summary_df = pd.DataFrame([results['summary']])
        summary_file = f'{output_prefix}_summary.csv'
        summary_df.to_csv(summary_file, index=False)
        print(f"✓ Сводка экспортирована в {summary_file}")
        
        
    def create_solution_archive(self, results: Dict, archive_name: str = 'solution.zip'):
        """Создание zip архива с файлом решения"""
        if results is None:
            print("Нет результатов для архивирования")
            return
        
        solution_file = 'solution_paths.csv'
        
        solution_data = []
        
        for flow in results['flows']:
            if flow['flow'] > 0.001:
                solution_data.append({
                    'src': flow['src'],
                    'dst': flow['dst'],
                    'volume': flow['flow'],
                    'path_nodes': flow['path']
                })
        
        solution_data.sort(key=lambda x: (x['src'], x['dst']))
        
        solution_df = pd.DataFrame(solution_data)
        solution_df.to_csv(solution_file, index=False)
        
        with zipfile.ZipFile(archive_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
            zipf.write(solution_file, arcname=solution_file)
        
        os.remove(solution_file)
        
        print(f"✓ Архив с решением создан: {archive_name}")
        
        return archive_name

In [ ]:
nodes = 50
time_limit_seconds = 3600
max_paths_length_2 = 75
solver_profile = 'highs'
data_folder = f'dataset/{nodes}_nodes'

distance_matrix_file = os.path.join(data_folder, 'distance_matrix.csv')
offices_file = os.path.join(data_folder, 'offices.csv')
reqs_file = os.path.join(data_folder, 'reqs.csv')



print(f"\nПараметры:")
print(f"  Узлов в графе: {nodes}")
print(f"  Лимит времени: {time_limit_seconds}s")
print(f"  Макс. путей длины 2: {max_paths_length_2}")
print(f"  Профиль решателя: {solver_profile}")

solver = MultiCommodityFlowSolver(
    distance_matrix_file=distance_matrix_file,
    offices_file=offices_file,
    reqs_file=reqs_file,
    max_paths_length_2=max_paths_length_2,
    solver_profile=solver_profile
)

if not solver.build_model():
    print("Ошибка при построении модели")
    exit()

if not solver.solve(time_limit_seconds=time_limit_seconds):
    print("Не удалось найти решение")
    exit()

results = solver.get_results()
solver.create_solution_archive(
    results, 
    archive_name=f'2k_solutions/solution_paths_{nodes}_tl_{time_limit_seconds}_max_paths_length_2_{max_paths_length_2}_cvxpy_{solver_profile}.csv.zip'
)